# Train ViT-B Stacking Detector

In [ ]:
%reload_ext autoreload
%autoreload 2

%matplotlib inline

import gc
import json
import os
import random
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
configured_root = os.environ.get("CODING_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    candidates = [CURRENT, CURRENT.parent]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").exists()),
        CURRENT,
    )

%cd {PROJECT_ROOT}

os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / "cache" / "hf"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "cache" / "matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from data_pipeline.image_io import load_image, load_image_safe, resolve_path
from data_pipeline.manifest import Blocklist, load as load_manifest, validate
from evaluation.metrics import compute
from evaluation.visualization import reliability_diagram, use_house_style
from modeling.common.device import device_report, get_device

use_house_style()


def progress(percent: float, stage: str, detail: str = "") -> None:
    """One consistent progress line for every notebook stage."""
    suffix = f" — {detail}" if detail else ""
    print(f"[PIPELINE {percent:6.1f}%] {stage}{suffix}", flush=True)


progress(2, "Setting", "project imports and autoreload are ready")

In [ ]:
TEST_MODE = False
MODEL_MODE = "saved"
RANDOM_SEED = 1337

BACKBONE = "vit_base_patch16_dinov3.lvd1689m"
CROPS = 3
CROP_SIZE = 224

# Deterministic variants preserve clean/degraded crop alignment.
TRAIN_VARIANTS = ["jpeg=50", "blur=1.0"]

# Expert probabilities are additional head inputs.
EXPERTS = ["community_forensics"]

SMOKE_TRAIN_PER_CLASS = 60
SMOKE_VALIDATION_PER_CLASS = 40
SMOKE_TEST_PER_CLASS = 40

HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.1
FIT_EPOCHS = 30
FIT_BATCH_SIZE = 1024
FIT_LR = 3e-4

# Fixed before test evaluation to avoid leakage.
BRANCH_A_WEIGHT = 0.70
BRANCH_B_WEIGHT = 0.30

MIN_CONVERGENCE_EPOCHS = 3
TARGET_ROBUST_AUC = 0.85
MAX_GENERALIZATION_GAP = 0.10
TARGET_TEST_CORRECT_PERCENT = 80.0
TARGET_TPR_AT_1_FPR_PERCENT = 30.0
MAX_VALIDATION_ECE = 0.10

assert MODEL_MODE in {"train", "saved"}
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = get_device()

# Fail early if this environment cannot construct DINOv3.
import timm

if not timm.is_model(BACKBONE.split(".")[0]):
    raise RuntimeError(
        f"This kernel cannot load {BACKBONE!r}.\n"
        f"  interpreter : {sys.executable}\n"
        f"  timm        : {timm.__version__} (needs >= 1.0 for DINOv3)\n\n"
        "Switch the notebook kernel to 'AIGC Detect (.venv)'. In VS Code that is "
        "the kernel picker at the top right of the notebook."
    )
RUN_NAME = "notebook_vitb_smoke" if TEST_MODE else "notebook_vitb"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "stacking" / RUN_NAME
CHECKPOINT_PATH = WEIGHTS_DIR / "best_robust.pt"
CALIBRATION_PATH = PROJECT_ROOT / "weights" / "fusion" / f"{RUN_NAME}_temperature.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_ROOT = PROJECT_ROOT / "cache" / "features"
SCORE_ROOT = PROJECT_ROOT / "cache" / "scores"

settings = pd.DataFrame(
    {
        "setting": [
            "TEST_MODE", "MODEL_MODE", "device", "timm", "run name", "backbone",
            "crops", "train variants", "experts",
            "Branch A' checkpoint", "calibration", "output directory",
        ],
        "value": [
            TEST_MODE, MODEL_MODE, str(DEVICE), timm.__version__, RUN_NAME, BACKBONE,
            f"{CROPS} x {CROP_SIZE}px (native, no resize)",
            ", ".join(TRAIN_VARIANTS), ", ".join(EXPERTS) or "none",
            str(CHECKPOINT_PATH), str(CALIBRATION_PATH), str(OUTPUT_DIR),
        ],
    }
)
display(settings)
display(pd.DataFrame([device_report()]))
progress(5, "Setting", "user controls validated")

In [ ]:
pretrained_status = pd.DataFrame(
    [
        {
            "asset": "Branch A' head checkpoint",
            "expected location": str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
            "available": CHECKPOINT_PATH.exists(),
        },
        {
            "asset": "Hugging Face cache (backbone + released detector)",
            "expected location": str(Path(os.environ["HF_HOME"]).relative_to(PROJECT_ROOT)),
            "available": Path(os.environ["HF_HOME"]).exists(),
        },
        {
            "asset": "Frozen-feature cache",
            "expected location": str(FEATURE_ROOT.relative_to(PROJECT_ROOT)),
            "available": FEATURE_ROOT.exists(),
        },
        {
            "asset": "Expert-score cache",
            "expected location": str(SCORE_ROOT.relative_to(PROJECT_ROOT)),
            "available": SCORE_ROOT.exists(),
        },
    ]
)
display(pretrained_status)

In [ ]:
progress(8, "Data Input", "loading train, validation, benchmark, and test manifests")

MANIFEST_PATHS = {
    "train": PROJECT_ROOT / "data/train/manifest.csv",
    "validation": PROJECT_ROOT / "data/evaluation/validation_manifest.csv",
    "benchmark": PROJECT_ROOT / "data/evaluation/benchmark_manifest.csv",
    "test": PROJECT_ROOT / "data/test/manifest.csv",
}

official = {name: load_manifest(path) for name, path in MANIFEST_PATHS.items()}

for name, frame in official.items():
    print(f"\n{name.upper()} — {len(frame):,} rows")
    display(frame.head(5))

progress(12, "Data Input", "official manifest head(5) tables displayed")

In [ ]:
def balanced_sample(frame: pd.DataFrame, per_class: int, seed: int) -> pd.DataFrame:
    pieces = []
    for label, group in frame.groupby("label"):
        pieces.append(group.sample(min(per_class, len(group)), random_state=seed + int(label)))
    return pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


# Cache indices depend on immutable manifest row order.
if TEST_MODE:
    notebook_cache = PROJECT_ROOT / "cache/notebook/manifests"
    notebook_cache.mkdir(parents=True, exist_ok=True)
    ACTIVE_MANIFESTS = {
        "train": notebook_cache / "vitb_train_smoke.csv",
        "validation": notebook_cache / "vitb_validation_smoke.csv",
        "test": notebook_cache / "vitb_test_smoke.csv",
    }
    for name, per_class, offset in [
        ("train", SMOKE_TRAIN_PER_CLASS, 0),
        ("validation", SMOKE_VALIDATION_PER_CLASS, 10),
        ("test", SMOKE_TEST_PER_CLASS, 20),
    ]:
        if not ACTIVE_MANIFESTS[name].exists():
            balanced_sample(official[name], per_class, RANDOM_SEED + offset).to_csv(
                ACTIVE_MANIFESTS[name], index=False
            )
else:
    plan_dir = PROJECT_ROOT / "data/plan"
    ACTIVE_MANIFESTS = {
        "train": plan_dir / "train.csv",
        "validation": plan_dir / "validation.csv",
        "test": plan_dir / "test.csv",
    }
    missing = [str(p) for p in ACTIVE_MANIFESTS.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Full mode reads the balanced subsamples written by the caching plan. "
            "Run this first:\n"
            "  python -m workflows.cache_plan --config configs/cache_plan.vitb.yaml\n"
            f"Missing: {missing}"
        )

active_train = load_manifest(ACTIVE_MANIFESTS["train"])
active_validation = load_manifest(ACTIVE_MANIFESTS["validation"])
active_test = load_manifest(ACTIVE_MANIFESTS["test"])

active_counts = pd.concat(
    [
        frame.groupby("label").size().rename(name)
        for name, frame in {
            "train": active_train,
            "validation": active_validation,
            "test": active_test,
        }.items()
    ],
    axis=1,
).fillna(0).astype(int)
active_counts.index = active_counts.index.map({0: "authentic (0)", 1: "AI-generated (1)"})

print(f"Mode: {'SMALL END-TO-END TEST' if TEST_MODE else 'FULL CACHED DATA'}")
display(active_counts)
print("\nActive training rows:")
display(active_train.head(5))
progress(18, "Data Input", f"active train={len(active_train):,}, val={len(active_validation):,}, test={len(active_test):,}")

In [ ]:
progress(20, "Data Preparation", "checking paths, duplicates, classes, and quarantine")

blocklist = Blocklist.load()


def preparation_report(name: str, frame: pd.DataFrame) -> dict[str, object]:
    paths = frame["path"].astype(str)
    return {
        "dataset": name,
        "rows": len(frame),
        "missing files": sum(not resolve_path(path).exists() for path in paths),
        "duplicate sha256": int(frame.duplicated("sha256").sum()),
        "exact benchmark collisions": int(frame["sha256"].isin(blocklist.sha256).sum()),
        "authentic": int((frame["label"] == 0).sum()),
        "AI-generated": int((frame["label"] == 1).sum()),
    }


reports = pd.DataFrame(
    [
        preparation_report("train", active_train),
        preparation_report("validation", active_validation),
        preparation_report("test", active_test),
    ]
)
display(reports)

validate(active_train, blocklist=blocklist, check_phash=False)
validate(active_validation, blocklist=blocklist, check_phash=False)
validate(active_test, blocklist=blocklist, check_phash=False)
progress(28, "Data Preparation", "cleaning and quarantine audit passed")

In [ ]:
from modeling.common.degradations import Cell, apply_chain

review_cells = [
    ("clean", None),
    ("JPEG quality 50", Cell("jpeg", 50)),
    ("blur sigma 1", Cell("blur", 1.0)),
    ("noise sigma .05", Cell("noise", 0.05)),
    ("center crop 80%", Cell("crop", 0.80)),
]

review_rows = pd.concat(
    [active_train[active_train["label"] == label].head(1) for label in (0, 1)],
    ignore_index=True,
)

fig, axes = plt.subplots(len(review_rows), len(review_cells), figsize=(16, 6), squeeze=False)
for row_index, row in review_rows.iterrows():
    original = load_image(row["path"])
    for column_index, (title, cell) in enumerate(review_cells):
        image = original if cell is None else apply_chain(
            original, [cell], rng=np.random.default_rng(RANDOM_SEED)
        )
        axes[row_index, column_index].imshow(image)
        axes[row_index, column_index].set_title(
            f"{title}\nlabel={int(row['label'])}", fontsize=10
        )
        axes[row_index, column_index].axis("off")
fig.suptitle(
    "Transformation review — authentic row first, AI-generated row second",
    fontsize=14,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "transformation_review.png", dpi=150, bbox_inches="tight")
plt.show()
progress(35, "Data Preparation", "sample transformations reviewed and saved")

In [ ]:
from workflows import cache_features, cache_scores

progress(38, "Data Modelling", "building or reusing the frozen-feature cache")

ALL_VARIANTS = ["clean", *TRAIN_VARIANTS]
CACHE_SPLITS = {
    "train": (ACTIVE_MANIFESTS["train"], ALL_VARIANTS),
    "validation": (ACTIVE_MANIFESTS["validation"], ALL_VARIANTS),
    "test": (ACTIVE_MANIFESTS["test"], ["clean"]),
}

for split_name, (manifest, variants) in CACHE_SPLITS.items():
    for variant in variants:
        code_ = cache_features.main([
            "--manifest", str(manifest),
            "--backbone", BACKBONE,
            "--crops", str(CROPS),
            "--crop-size", str(CROP_SIZE),
            "--variant", variant,
            "--batch-size", "8",
            "--workers", "4",
            "--out", str(FEATURE_ROOT),
        ])
        assert code_ == 0, f"feature caching failed for {split_name}/{variant}"
    for expert in EXPERTS:
        for variant in variants:
            code_ = cache_scores.main([
                "--manifest", str(manifest),
                "--model", expert,
                "--variant", variant,
                "--batch-size", "16",
                "--out", str(SCORE_ROOT),
            ])
            assert code_ == 0, f"score caching failed for {split_name}/{expert}/{variant}"

progress(45, "Data Modelling", "feature and expert-score caches ready")

In [ ]:
from modeling.stacking.features import FeatureSet
from workflows.fit_stacker import expert_table

def build_feature_set(split: str) -> FeatureSet:
    manifest, variants = CACHE_SPLITS[split]
    return FeatureSet(
        manifest,
        backbone=BACKBONE,
        crops=CROPS,
        crop_size=CROP_SIZE,
        variants=variants,
        root=FEATURE_ROOT,
        extra=expert_table(manifest, EXPERTS, variants, SCORE_ROOT),
    )


train_features = build_feature_set("train")
validation_features = build_feature_set("validation")
test_features = build_feature_set("test")

# Keep checkpoint selection separate from temperature calibration.
selection_features, calibration_features = validation_features.split(0.5, seed=RANDOM_SEED)

print("TRAIN")
display(train_features.summary())
print("VALIDATION")
display(validation_features.summary())
print(
    f"validation split -> {len(selection_features):,} rows select the checkpoint, "
    f"{len(calibration_features):,} rows fit the temperature"
)

cache_size_mb = sum(
    path.stat().st_size for path in FEATURE_ROOT.rglob("*.npy")
) / 1e6
print(f"\nfeature cache on disk: {cache_size_mb:,.1f} MB")
progress(48, "Data Modelling", "cached features loaded")

In [ ]:
from modeling.dinov3.losses import LossWeights
from modeling.stacking.head import (
    FitConfig, StackingConfig, evaluate, fit, history_frame, load, save,
)

training_result = None

if MODEL_MODE == "train":
    progress(50, "Branch A'", "fitting the head on cached features")
    stacking_config = StackingConfig(
        dim=train_features.dim,
        extra_dim=train_features.extra_dim,
        head_hidden=HEAD_HIDDEN,
        head_dropout=HEAD_DROPOUT,
        feature_correction=True,
        expert_names=list(EXPERTS),
    )
    fit_config = FitConfig(
        epochs=FIT_EPOCHS,
        batch_size=FIT_BATCH_SIZE,
        lr=FIT_LR,
        seed=RANDOM_SEED,
        loss=LossWeights(focal_gamma=2.0, focal_alpha=0.5, cls_degraded=1.0, kl=0.5, feat_mse=0.25),
    )

    def show(epoch: int, train_row: dict, summary: dict) -> None:
        if summary and (epoch + 1) % max(1, FIT_EPOCHS // 10) == 0:
            print(
                f"  epoch {epoch + 1:3d}/{FIT_EPOCHS}"
                f"  loss {train_row.get('train_total', float('nan')):.4f}"
                f"  clean {summary['clean_auc']:.4f}"
                f"  robust {summary['robust_auc']:.4f}"
                f"  worst {summary['worst_cell_auc']:.4f}",
                flush=True,
            )

    import time as _time
    _started = _time.time()
    training_result = fit(train_features, selection_features, stacking_config, fit_config, progress=show)
    branch_a_head = training_result["model"]
    fit_seconds = _time.time() - _started

    save(
        CHECKPOINT_PATH, branch_a_head,
        backbone=BACKBONE, crops=CROPS, crop_size=CROP_SIZE,
        history=training_result["history"],
        best_robust_auc=training_result["best_robust_auc"],
        variants=ALL_VARIANTS,
    )
    history_frame(training_result["history"]).to_csv(
        CHECKPOINT_PATH.with_suffix(".history.csv"), index=False
    )
    print(f"\nfitted in {fit_seconds:.1f}s on {len(train_features):,} images "
          f"({len(train_features) * CROPS:,} cached crops)")
    print(f"saved {CHECKPOINT_PATH}")
    print(f"saved {CHECKPOINT_PATH.with_suffix('.history.csv')}")
    progress(58, "Branch A'", f"head fitted in {fit_seconds:.1f}s")
else:
    progress(50, "Branch A'", "saved-parameter mode selected")
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Missing {CHECKPOINT_PATH}. Set MODEL_MODE='train', or copy a fitted head here."
        )
    branch_a_head, _state = load(CHECKPOINT_PATH, map_location=str(DEVICE))
    fit_seconds = float("nan")
    progress(58, "Branch A'", f"using saved head {CHECKPOINT_PATH}")

branch_a_head = branch_a_head.to(DEVICE).eval()
print(branch_a_head.describe())

In [ ]:
from modeling.registry import check_param_budget, get
from modeling.stacking.scorer import StackingScorer

branch_a_scorer = StackingScorer(
    checkpoint=str(CHECKPOINT_PATH),
    device=str(DEVICE),
    n_crops=CROPS,
    seed=RANDOM_SEED,
)
branch_a_params = check_param_budget(branch_a_scorer)
print(branch_a_scorer.describe())
print(f"\nBranch A' ready: {branch_a_params:,} inference parameters "
      f"({100 * branch_a_params / 2_000_000_000:.1f}% of the 2B limit)")
progress(60, "Branch A'", "inference scorer constructed")

In [ ]:
progress(62, "Branch B", "loading Community Forensics released checkpoint")

branch_b_name = "community_forensics"
branch_b_scorer = get(branch_b_name, device=str(DEVICE))
branch_b_params = check_param_budget(branch_b_scorer)
print(f"Branch B ready: {branch_b_name}, {branch_b_params:,} inference parameters")
progress(66, "Branch B", "released branch ready")

In [ ]:
from modeling.stacking.head import predict_variant
from workflows.cache_scores import load_scores


def branch_a_cached(feature_set: FeatureSet, variant: str = "clean") -> np.ndarray:
    return predict_variant(branch_a_head, feature_set, variant, device=DEVICE)


def branch_b_cached(manifest: Path, rows: np.ndarray, variant: str = "clean") -> np.ndarray:
    scores, done = load_scores(manifest, model=branch_b_name, variant=variant, root=SCORE_ROOT)
    if not done.all():
        print(f"WARNING: {int((~done).sum()):,} rows missing Branch B scores; using the 0.5 fallback")
    return scores[rows]


progress(68, "Validation scoring", "Branch A' from cached features")
validation_a = branch_a_cached(calibration_features)
progress(71, "Validation scoring", "Branch B from cached scores")
validation_b = branch_b_cached(ACTIVE_MANIFESTS["validation"], calibration_features.index)
validation_labels = calibration_features.labels

print(f"scored {len(validation_a):,} held-out validation rows for calibration")
display(
    pd.DataFrame([
        {"branch": "A' (frozen ViT-B + head)", **compute(validation_labels, validation_a)},
        {"branch": f"B ({branch_b_name})", **compute(validation_labels, validation_b)},
    ])[["branch", "roc_auc", "average_precision", "tpr_at_fpr0.01", "tpr_at_fpr0.05"]].round(4)
)

In [ ]:
from modeling.ensemble.calibration import TemperatureCalibrator

valid_mask = np.isfinite(validation_a) & np.isfinite(validation_b)
validation_raw_fused = (
    BRANCH_A_WEIGHT * validation_a[valid_mask]
    + BRANCH_B_WEIGHT * validation_b[valid_mask]
) / (BRANCH_A_WEIGHT + BRANCH_B_WEIGHT)
validation_labels = validation_labels[valid_mask]

if MODEL_MODE == "saved" and CALIBRATION_PATH.exists():
    calibrator = TemperatureCalibrator.load(CALIBRATION_PATH)
    calibration_action = "loaded"
else:
    calibrator = TemperatureCalibrator().fit(validation_raw_fused, validation_labels)
    calibrator.save(CALIBRATION_PATH)
    calibration_action = "fitted on the held-out validation half"

validation_calibrated = calibrator.transform(validation_raw_fused)
print(
    f"Calibration {calibration_action}: T={calibrator.temperature:.6f} "
    f"-> {CALIBRATION_PATH}"
)
progress(77, "Fusion + calibration", "weights and temperature frozen before test")

In [ ]:
progress(79, "Test scoring", "Branch A' from cached features")
test_a = branch_a_cached(test_features)
progress(84, "Test scoring", "Branch B from cached scores")
test_b = branch_b_cached(ACTIVE_MANIFESTS["test"], test_features.index)

test_rows = active_test.iloc[test_features.index]
results = test_rows[["path", "label", "source", "generator"]].copy().reset_index(drop=True)
results["branch_a"] = test_a
results["branch_b"] = test_b
results["fused_raw"] = (
    BRANCH_A_WEIGHT * results["branch_a"] + BRANCH_B_WEIGHT * results["branch_b"]
) / (BRANCH_A_WEIGHT + BRANCH_B_WEIGHT)
results["pred"] = calibrator.transform(results["fused_raw"].to_numpy())

metric_rows = []
labels_array = results["label"].to_numpy(dtype=int)
for model_name, column in (
    ("Branch A' — frozen ViT-B + head", "branch_a"),
    (f"Branch B — {branch_b_name}", "branch_b"),
    ("Fused + calibrated", "pred"),
):
    scores = results[column].to_numpy()
    predicted = (scores >= 0.5).astype(int)
    correct = predicted == labels_array
    authentic = labels_array == 0
    generated = labels_array == 1
    metric_rows.append({
        "model": model_name,
        **compute(labels_array, scores),
        "pred_correct_percent": 100.0 * correct.mean(),
        "authentic_correct_percent": 100.0 * correct[authentic].mean(),
        "ai_generated_correct_percent": 100.0 * correct[generated].mean(),
        "correct_predictions": int(correct.sum()),
        "total_predictions": int(len(correct)),
    })
metrics_table = pd.DataFrame(metric_rows)

results["predicted_label"] = (results["pred"] >= 0.5).astype(int)
results["correct"] = results["predicted_label"] == results["label"]
fused_correct = int(results["correct"].sum())
fused_correct_percent = 100.0 * fused_correct / max(1, len(results))

display(
    metrics_table[
        [
            "model", "pred_correct_percent", "authentic_correct_percent",
            "ai_generated_correct_percent", "roc_auc", "average_precision",
            "tpr_at_fpr0.01", "tpr_at_fpr0.05",
        ]
    ].rename(columns={
        "pred_correct_percent": "Pred correct (%)",
        "authentic_correct_percent": "Authentic correct (%)",
        "ai_generated_correct_percent": "AI-generated correct (%)",
    }).round(2)
)
print(
    f"Final fused predictions correct: {fused_correct:,}/{len(results):,} "
    f"= {fused_correct_percent:.2f}%"
)
display(results.head(5))

csv_path = OUTPUT_DIR / "test_predictions_with_labels.csv"
json_path = OUTPUT_DIR / "predictions.json"
metrics_path = OUTPUT_DIR / "metrics.csv"
results.to_csv(csv_path, index=False)
metrics_table.to_csv(metrics_path, index=False)
json_records = [
    {"image_path": path, "pred": float(prediction)}
    for path, prediction in zip(results["path"], results["pred"])
]
json_path.write_text(json.dumps(json_records, indent=2) + "\n")
progress(88, "Data Modelling", f"predictions and metrics saved under {OUTPUT_DIR}")

In [ ]:
import shutil

demo_dir = OUTPUT_DIR / "demo_images"
if demo_dir.exists():
    shutil.rmtree(demo_dir)
demo_dir.mkdir(parents=True)

demo_rows = pd.concat(
    [results[results["label"] == label].head(3) for label in (0, 1)], ignore_index=True
)
for position, row in demo_rows.iterrows():
    source = resolve_path(row["path"])
    shutil.copy(source, demo_dir / f"{position:02d}_label{int(row['label'])}{source.suffix}")

from workflows.predict import predict_dir

live_records = predict_dir(
    demo_dir,
    model="stacking",
    checkpoint=str(CHECKPOINT_PATH),
    device=str(DEVICE),
    batch_size=8,
    quiet=True,
)
live = pd.DataFrame(live_records)
live["file"] = live["image_path"].map(lambda p: Path(p).name)
live["true label"] = live["file"].str.extract(r"label(\d)").astype(int)
display(live[["file", "true label", "pred"]].round(4))
print(
    "The live scorer runs the backbone and the released expert on raw images; the "
    "cached path above reuses stored crops, so the two agree in ranking rather than "
    "to the last decimal."
)
progress(90, "Data Modelling", "live predict.py path verified")

In [ ]:
history = training_result["history"] if training_result is not None else []
if not history and CHECKPOINT_PATH.exists():
    history = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False).get("history", [])

history_table = history_frame(history)

loss_columns = [
    column for column in (
        "train_total", "train_cls_clean", "train_cls_degraded", "train_kl", "train_feat_mse"
    )
    if column in history_table and history_table[column].notna().any()
]

if loss_columns:
    step = max(1, len(history_table) // 10)
    loss_milestones = (
        history_table.assign(epoch=history_table["epoch"] + 1)
        .loc[lambda frame: frame["epoch"] % step == 0, ["epoch", *loss_columns]]
        .copy()
    )
    display(loss_milestones.round(6))

    fig, ax = plt.subplots(figsize=(9, 5))
    for column in loss_columns:
        ax.plot(
            loss_milestones["epoch"],
            loss_milestones[column],
            marker="o",
            label=column.removeprefix("train_").replace("_", " "),
        )
    ax.set_title(f"Head convergence — every {step} epoch(s)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Average loss (lower is better)")
    ax.legend(title="Loss component")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "training_loss_convergence.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("This checkpoint has no persisted epoch-loss history.")
progress(92, "Visualization", "training-loss statistics reviewed")

In [ ]:
auc_columns = [
    column for column in ("clean_auc", "robust_auc", "worst_cell_auc")
    if column in history_table and history_table[column].notna().any()
]
if auc_columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for column in auc_columns:
        axes[0].plot(
            history_table["epoch"] + 1,
            history_table[column],
            marker="o",
            label=column.replace("_", " "),
        )
    axes[0].axhline(
        TARGET_ROBUST_AUC, color="#16a34a", linestyle="--", label="robust AUC target"
    )
    axes[0].set_title("Branch A' validation AUC while fitting the head")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("AUC (higher is better)")
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()

    axes[1].plot(
        history_table["epoch"] + 1, history_table["gap"], marker="o", color="#dc2626"
    )
    axes[1].axhline(
        MAX_GENERALIZATION_GAP,
        color="#16a34a",
        linestyle="--",
        label="maximum target gap",
    )
    axes[1].axhline(0, color="#64748b", linewidth=1)
    axes[1].set_title("Clean-to-degraded generalisation gap")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Clean AUC - robust AUC (closer to 0 is better)")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "validation_convergence.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No robust-validation history is available for this checkpoint.")
progress(94, "Visualization", "validation convergence and gap reviewed")

In [ ]:
correctness_plot = metrics_table.set_index("model")[[
    "pred_correct_percent", "authentic_correct_percent", "ai_generated_correct_percent"
]].rename(columns={
    "pred_correct_percent": "Overall",
    "authentic_correct_percent": "Authentic",
    "ai_generated_correct_percent": "AI-generated",
})
fig, ax = plt.subplots(figsize=(11, 5.5))
correctness_plot.plot(kind="bar", ax=ax, color=["#0f766e", "#3b82f6", "#f97316"])
ax.axhline(
    TARGET_TEST_CORRECT_PERCENT, color="#16a34a", linestyle="--", label="project target"
)
ax.set_title("Test predictions correct — overall and by ground-truth class")
ax.set_xlabel("Model output")
ax.set_ylabel("Correct predictions (%)")
ax.set_ylim(0, 102)
ax.tick_params(axis="x", rotation=10)
ax.legend(title="Correctness group", ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_percent_correct.png", dpi=150, bbox_inches="tight")
plt.show()
progress(96, "Visualization", "test correctness reviewed")

In [ ]:
_, ece_before = reliability_diagram(validation_labels, validation_raw_fused)
plt.suptitle("Validation reliability before temperature calibration", y=1.02)
plt.show()

_, ece_after = reliability_diagram(
    validation_labels,
    validation_calibrated,
    path=OUTPUT_DIR / "validation_reliability_after.png",
)
plt.suptitle("Validation reliability after temperature calibration", y=1.02)
plt.show()
print(f"Expected calibration error: {ece_before:.4f} -> {ece_after:.4f}")
progress(97, "Visualization", "calibration reliability reviewed")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
score_rng = np.random.default_rng(RANDOM_SEED)
for ax, (title, column) in zip(
    axes,
    [
        ("Branch A' score distribution", "branch_a"),
        ("Branch B score distribution", "branch_b"),
        ("Fused calibrated distribution", "pred"),
    ],
):
    class_scores = [
        results.loc[results["label"] == label, column].dropna().to_numpy()
        for label in (0, 1)
    ]
    boxes = ax.boxplot(
        class_scores,
        positions=[0, 1],
        orientation="horizontal",
        widths=0.42,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "#14191b", "linewidth": 2},
    )
    for patch, colour in zip(boxes["boxes"], ("#3b82f6", "#f97316")):
        patch.set_facecolor(colour)
        patch.set_alpha(0.28)
    for label, (values, colour) in enumerate(zip(class_scores, ("#3b82f6", "#f97316"))):
        shown = values if len(values) <= 500 else score_rng.choice(values, 500, replace=False)
        jitter = score_rng.normal(label, 0.055, size=len(shown))
        ax.scatter(shown, jitter, s=18, alpha=0.45, color=colour, edgecolors="none")
        if len(values):
            ax.text(
                float(np.median(values)), label + 0.28, f"median {np.median(values):.2f}",
                ha="center", va="bottom", fontsize=8, color=colour,
            )
    ax.axvline(0.5, color="#475569", linestyle="--", linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Predicted P(AI-generated)")
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6))
axes[0].set_yticks([0, 1], labels=["Authentic", "AI-generated"])
fig.suptitle("Score separation — farther apart is better", fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

fused_metrics = metrics_table.loc[metrics_table["model"] == "Fused + calibrated"].iloc[0]
enough_epochs = len(history_table) >= MIN_CONVERGENCE_EPOCHS
loss_decreased = (
    enough_epochs
    and "train_total" in history_table
    and history_table["train_total"].iloc[-1] < history_table["train_total"].iloc[0]
)
best_robust_auc = history_table.get("robust_auc", pd.Series(dtype=float)).max()
latest_gap = (
    float(history_table["gap"].dropna().iloc[-1])
    if "gap" in history_table and history_table["gap"].notna().any()
    else float("nan")
)

readiness_rows = [
    {
        "quality question": "Did the head's loss decrease across enough epochs?",
        "observed": (
            f"{history_table['train_total'].iloc[0]:.4f} -> "
            f"{history_table['train_total'].iloc[-1]:.4f}"
            if "train_total" in history_table and not history_table.empty else "unavailable"
        ),
        "target": f"decrease across >= {MIN_CONVERGENCE_EPOCHS} epochs",
        "status": "PASS" if loss_decreased else (
            "NOT ASSESSABLE" if not enough_epochs else "FAIL"
        ),
    },
    {
        "quality question": "Does Branch A' work across degradations?",
        "observed": f"best robust AUC {best_robust_auc:.3f}",
        "target": f">= {TARGET_ROBUST_AUC:.2f}",
        "status": (
            "PASS"
            if pd.notna(best_robust_auc) and best_robust_auc >= TARGET_ROBUST_AUC
            else "FAIL"
        ),
    },
    {
        "quality question": "Is the clean-to-degraded gap controlled?",
        "observed": f"gap {latest_gap:.3f}",
        "target": f"<= {MAX_GENERALIZATION_GAP:.2f}",
        "status": (
            "PASS"
            if pd.notna(latest_gap) and latest_gap <= MAX_GENERALIZATION_GAP
            else "FAIL"
        ),
    },
    {
        "quality question": "Are enough test predictions correct?",
        "observed": f"{fused_metrics['pred_correct_percent']:.2f}%",
        "target": f">= {TARGET_TEST_CORRECT_PERCENT:.1f}%",
        "status": (
            "PASS"
            if fused_metrics["pred_correct_percent"] >= TARGET_TEST_CORRECT_PERCENT
            else "FAIL"
        ),
    },
    {
        "quality question": "Does it catch generated images at 1% FPR?",
        "observed": f"{100 * fused_metrics['tpr_at_fpr0.01']:.2f}%",
        "target": f">= {TARGET_TPR_AT_1_FPR_PERCENT:.1f}%",
        "status": (
            "PASS"
            if 100 * fused_metrics["tpr_at_fpr0.01"] >= TARGET_TPR_AT_1_FPR_PERCENT
            else "FAIL"
        ),
    },
    {
        "quality question": "Are validation probabilities calibrated?",
        "observed": f"ECE {ece_after:.3f}",
        "target": f"<= {MAX_VALIDATION_ECE:.2f}",
        "status": "PASS" if ece_after <= MAX_VALIDATION_ECE else "FAIL",
    },
]
readiness_table = pd.DataFrame(readiness_rows)
status_colours = {"PASS": "#0E9384", "FAIL": "#A33A3E", "NOT ASSESSABLE": "#9A6407"}
fig, ax = plt.subplots(figsize=(12, 6.2))
ax.set_xlim(0, 1)
ax.set_ylim(-0.7, len(readiness_table) - 0.3)
ax.axis("off")
for row_index, row in readiness_table.iloc[::-1].reset_index(drop=True).iterrows():
    display_status = "NOT ASSESSABLE" if TEST_MODE else row["status"]
    colour = status_colours[display_status]
    ax.scatter(0.035, row_index, s=260, color=colour, clip_on=False)
    ax.text(0.035, row_index, "✓" if display_status == "PASS" else "!",
            ha="center", va="center", color="white", fontweight="bold")
    ax.text(0.075, row_index + 0.13, row["quality question"],
            ha="left", va="center", fontsize=10, fontweight="bold")
    ax.text(0.075, row_index - 0.16, f"Observed: {row['observed']}   •   Target: {row['target']}",
            ha="left", va="center", fontsize=9, color="#4A5459")
    ax.text(0.97, row_index, display_status, ha="right", va="center",
            fontsize=9, fontweight="bold", color=colour)
    if row_index < len(readiness_table) - 1:
        ax.hlines(row_index + 0.5, 0.075, 0.97, color="#E4E9E9", linewidth=1)
ax.set_title("Model-readiness checklist", loc="left", fontsize=14, pad=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "model_readiness.png", dpi=150, bbox_inches="tight")
plt.show()

if TEST_MODE:
    readiness_verdict = (
        "PIPELINE VALIDATED — quality/convergence not assessable from smoke data"
    )
elif (readiness_table["status"] == "PASS").all():
    readiness_verdict = "READY CANDIDATE — all configured quality gates passed"
else:
    readiness_verdict = (
        "NOT YET GOOD ENOUGH — review failed quality gates before release"
    )
print(f"\nMODEL READINESS: {readiness_verdict}")
progress(99, "Visualization", "convergence and readiness assessment complete")

In [ ]:
summary = {
    "test_mode": TEST_MODE,
    "model_mode": MODEL_MODE,
    "device": str(DEVICE),
    "backbone": BACKBONE,
    "crops_per_image": CROPS,
    "train_variants": ALL_VARIANTS,
    "branch_a_checkpoint": str(CHECKPOINT_PATH),
    "branch_a_inference_params": int(branch_a_params),
    "branch_b": branch_b_name,
    "head_fit_seconds": None if np.isnan(fit_seconds) else round(fit_seconds, 1),
    "feature_cache_mb": round(cache_size_mb, 1),
    "temperature": calibrator.temperature,
    "test_images_scored": len(results),
    "test_predictions_correct": f"{fused_correct}/{len(results)}",
    "test_predictions_correct_percent": round(fused_correct_percent, 2),
    "model_readiness": readiness_verdict,
    "predictions_json": str(json_path),
    "metrics_csv": str(metrics_path),
    "output_directory": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
display(pd.DataFrame(summary.items(), columns=["item", "value"]))
progress(100, "Complete", "full notebook workflow finished successfully")